# Task 1 — Article Type Classification

**Status:** planning scaffold only. There is no training code, selected model,
selected metric, fake result, or final claim in this notebook.

**Owner:** TODO(owner)


## 1. Task contract

Task 1 predicts the `articleType` of one fashion product from its image.

- **Input:** one fashion image from the provided dataset identified by its product `id`.
- **Prediction unit:** one product image.
- **Output:** one label from the fixed 124-class `articleType` vocabulary.
- **Final use:** the prediction will fill the `articleType` column in the final
  `id,gender,articleType,season,usage` submission file.
- **Development scope:** compare models using labelled development images and the
  five saved cross-validation folds.
- **Out of scope:** opening holdout labels, creating another split, using external
  images, or selecting the final submitted model before experiments are complete.
- **Main risks:** rare classes, visually similar article types, small images and
  possible product-family leakage.

The final submitted learned model will be trained from scratch. Pretrained weights
may only be used later as a clearly separated comparison benchmark.


## 2. Reading the data

Task 1 uses the teacher-image rows recorded in the shared manifest:

`data/processed/splits.csv`

The manifest must be loaded through `fashion.data.dataset.load_splits`. This keeps
holdout and quarantine labels hidden.

For Task 1, we keep rows that:

- belong to the `development` partition;
- have a valid `articleType` label;
- have an image path;
- use one of the 124 development-defined article-type labels.

We do not call `train_test_split`. The five `cv_fold` values created by Notebook 01
are the only allowed development splits.

This section loads metadata and image paths. It does not load image pixels yet.
Image decoding, resizing and normalization will be added after the preprocessing
comparison is defined.


In [1]:
from fashion.data.dataset import (
    get_samples,
    iter_cv_folds,
    load_label_maps,
    load_splits,
)

TARGET = "articleType"

# Safe loader: protected holdout and quarantine labels remain hidden.
splits = load_splits()

# Labelled Task 1 development rows only.
task1_development = get_samples(
    splits,
    partition="development",
    target=TARGET,
)

# Vocabulary created from development data by Notebook 01.
article_type_map = load_label_maps()[TARGET]
article_type_classes = tuple(article_type_map["classes"])
label_to_index = article_type_map["label_to_index"]

NUM_CLASSES = len(article_type_classes)


In [2]:
protected = splits["partition"].isin(["holdout", "quarantine"])

assert set(task1_development["partition"]) == {"development"}
assert task1_development["has_articleType_label"].all()
assert task1_development["id"].is_unique
assert task1_development["path"].astype(str).str.strip().ne("").all()
assert splits.loc[protected, TARGET].eq("").all()
assert NUM_CLASSES == 124
assert set(task1_development[TARGET]) == set(article_type_classes)

print(f"Task 1 development products: {len(task1_development):,}")
print(f"Article-type classes: {NUM_CLASSES}")
print("Protected labels remain sealed.")


Task 1 development products: 32,773
Article-type classes: 124
Protected labels remain sealed.


In [3]:
import pandas as pd

fold_rows = []

for fold, training_rows, validation_rows in iter_cv_folds(splits):
    training_rows = get_samples(training_rows, target=TARGET)
    validation_rows = get_samples(validation_rows, target=TARGET)

    fold_rows.append(
        {
            "validation_fold": fold,
            "training_products": len(training_rows),
            "validation_products": len(validation_rows),
            "training_classes": training_rows[TARGET].nunique(),
            "validation_classes": validation_rows[TARGET].nunique(),
        }
    )

fold_summary = pd.DataFrame(fold_rows)
fold_summary


,validation_fold,training_products,validation_products,training_classes,validation_classes
0,0,26220,6553,123,102
1,1,26217,6556,120,108
2,2,26220,6553,122,111
3,3,26219,6554,122,107
4,4,26216,6557,121,109


## 3. Development-validation strategy

**CV mode: TODO(owner)**

Choose and record one mode before experiments:

- one fixed validation fold; or
- all five precomputed folds.

Give the reason, compute budget, and comparison rule. Do not run several folds
and report only the best-looking one.


## 4. Fixed-fold declaration

**Validation fold if fixed mode: TODO(owner)**

If all five folds are used, write `not applicable — all folds` and define how
fold results will be aggregated before seeing them.


## 5. Task-specific preprocessing and leakage rules

- Inputs available at prediction time: TODO(owner)
- Task transform pipeline: TODO(owner)
- Values learned from data: TODO(owner)
- Fit boundary: training folds of the current round only
- Validation application rule: TODO(owner; transform only, never refit)
- Holdout rule: sealed until Notebook 06 after every choice is frozen

If mean/std, rebalancing values, sampling rules, or feature selection are used,
fit them again inside each round's training rows. Do not copy a value fitted on
all development into a fold experiment.


## 6. Preprocessing comparisons to run

Define the alternatives that answer a real question for this task.

| Comparison ID | Question | Alternative A | Alternative B | Controlled variables | Evidence needed |
|---|---|---|---|---|---|
| TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) |

Do not choose an alternative in this scaffold.


## 7. Hypotheses and baseline

- Baseline purpose and implementation: TODO(owner)
- Hypothesis 1 and expected observation: TODO(owner)
- Hypothesis 2 and expected observation: TODO(owner)
- Long-tail taxonomy and imbalance question: TODO(owner)
- Sampling or loss-weighting options worth comparing: TODO(owner)
- Failure condition that would reject each hypothesis: TODO(owner)

A baseline is a comparison anchor, not a preselected winner.


## 8. Candidate model comparisons

Define candidate families only after checking the assignment constraints.

| Candidate ID | Why include it | Capacity/complexity control | Scratch-training compliance | Expected trade-off |
|---|---|---|---|---|
| TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) |

No candidate is selected in advance.


## 9. Metric selection and freeze

- Primary development metric: TODO(owner)
- Why it matches the task decision: TODO(owner)
- Secondary diagnostic measures: TODO(owner)
- Aggregation across classes/outputs/folds: TODO(owner)
- Freeze timestamp or decision-log entry: TODO(owner)

Choose and document the metric before comparing final candidates. Do not edit the
metric because a later result looks inconvenient.


## 10. Experiment matrix

| Run intent | CV mode/fold | Preprocessing ID | Candidate ID | Controlled seed/budget | Question answered |
|---|---|---|---|---|---|
| TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) |

Keep the matrix broad enough to justify choices, but small enough that each run
receives error and cost analysis.


## 11. Run registry

Every training/evaluation run must append through `fashion.train.registry` to
`results/runs.csv`.

- Run IDs: TODO(owner after execution)
- Checkpoint paths: TODO(owner after execution)
- Configuration hashes: TODO(owner after execution)
- Evidence tables/figures linked to run IDs: TODO(owner after execution)

Do not hand-type final comparison numbers into the report.


## 12. Error analysis

- Define useful error slices before viewing results: TODO(owner)
- Inspect representative successes and failures with IDs: TODO(owner)
- Check rare/ambiguous groups and family effects: TODO(owner)
- Define rare-class error slices before viewing results: TODO(owner)
- Separate data limitations from model limitations: TODO(owner)
- Record unexpected failure modes honestly: TODO(owner)


## 13. Robustness and efficiency

- Robustness questions and controlled tests: TODO(owner)
- Runtime and hardware measurement rule: TODO(owner)
- Memory/storage or index cost: TODO(owner)
- Stability across seeds/folds where applicable: TODO(owner)
- Practical deployment limitation: TODO(owner)


## 14. Decision log

| Decision | Evidence considered | Choice | Rejected alternatives | Limitation | Date/owner |
|---|---|---|---|---|---|
| TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) |

Record choices as they are made. Do not rewrite history after holdout access.


## 15. Handoff to final evaluation

Before Notebook 06, provide:

- frozen winning run ID(s): TODO(owner)
- frozen preprocessing configuration: TODO(owner)
- frozen metric definition and CV evidence: TODO(owner)
- refit procedure for all development: TODO(owner)
- expected final checkpoint/output path: TODO(owner)
- unresolved risks and honest limitations: TODO(owner)

**Handoff status: NOT READY — owner must complete every item above.**
